In [ ]:
# prompt: extract list of unique lat long from Training_data_uhi_index_2025-02-18.csv

import pandas as pd

# Assuming the CSV file is in the current directory.
# If not, provide the full path to the file.
file_path = 'Training_data_uhi_index_2025-02-18.csv'

try:
    df = pd.read_csv(file_path)

    # Extract unique latitude and longitude values.
    unique_locations = df[['Latitude', 'Longitude']].drop_duplicates()

    # Convert the result back to a list of dictionaries (optional).
    unique_locations_list = unique_locations.to_dict('records')

    print(len(unique_locations_list))

except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except KeyError as e:
    print(f"Error: Column not found: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


11229


In [ ]:
# prompt: determine a suitable number of batches for 11229 rows

import math

# Assuming 'unique_locations_list' is defined from the previous code.
num_rows = len(unique_locations_list)  # Or directly use len(df) if available.
batch_size = 256  # A reasonable batch size, adjust as needed

num_batches = math.ceil(num_rows / batch_size)
print(f"Number of batches for {num_rows} rows: {num_batches}")


Number of batches for 11229 rows: 44


In [ ]:
# prompt: def get_hourly_temperatures(latitude, longitude, hour, next_hour):
#     # Use only the date portion for the API call
#     date_str = hour.strftime('%Y-%m-%d')
#     url = (f"https://api.open-meteo.com/v1/forecast?"
#            f"latitude={latitude}&longitude={longitude}&hourly=temperature_2m&"
#            f"start_date={date_str}&end_date={date_str}&timezone=auto")
#     try:
#         # Delay added to help prevent rate limiting
#         time.sleep(5)
#         response = requests.get(url)
#         response.raise_for_status()
#         data = json.loads(response.text)
#         if 'hourly' in data and 'temperature_2m' in data['hourly']:
#             times = pd.to_datetime(data['hourly']['time'])
#             temps = data['hourly']['temperature_2m']
#             return dict(zip(times, temps))
#         else:
#             print(f"Error: Invalid response from API for lat {latitude}, lon {longitude} on {date_str}")
#             return None
#     except requests.exceptions.RequestException as e:
#         print(f"Error fetching temperature for {latitude}, {longitude}: {e}")
#         return None
# Use the lat long list from previous step to get the temperature for 21 July 2024 3PM and 4PM in New York. Do it in batches and save a csv for each batch. implement tqdm for progress tracking for each batch. Download the file after every batch completes and give me the ability to start from a specific batch if required

import pandas as pd
import math
import requests
import json
import time
from datetime import datetime
from tqdm import tqdm
from google.colab import files

# Assuming the CSV file is in the current directory.
# If not, provide the full path to the file.
file_path = 'Training_data_uhi_index_2025-02-18.csv'

try:
    df = pd.read_csv(file_path)

    # Extract unique latitude and longitude values.
    unique_locations = df[['Latitude', 'Longitude']].drop_duplicates()

    # Convert the result back to a list of dictionaries.
    unique_locations_list = unique_locations.to_dict('records')

    print(len(unique_locations_list))

except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except KeyError as e:
    print(f"Error: Column not found: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


def get_hourly_temperatures(latitude, longitude, hour, next_hour):
    # Use only the date portion for the API call
    date_str = hour.strftime('%Y-%m-%d')
    url = (f"https://api.open-meteo.com/v1/forecast?"
       f"latitude={latitude}&longitude={longitude}&hourly=temperature_2m&"
       f"start_date={date_str}&end_date={date_str}&timezone=America/New_York")

    try:
        # Delay added to help prevent rate limiting
        time.sleep(0.35)
        response = requests.get(url)
        response.raise_for_status()
        data = json.loads(response.text)
        if 'hourly' in data and 'temperature_2m' in data['hourly']:
            times = pd.to_datetime(data['hourly']['time'])
            temps = data['hourly']['temperature_2m']
            return dict(zip(times, temps))
        else:
            print(f"Error: Invalid response from API for lat {latitude}, lon {longitude} on {date_str}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching temperature for {latitude}, {longitude}: {e}")
        return None

# Parameters
num_rows = len(unique_locations_list)
batch_size = 128
num_batches = math.ceil(num_rows / batch_size)
start_batch = 2  # Change this to start from a different batch

hour = pd.Timestamp(2024, 7, 21, 15, 0, 0, tz='America/New_York')    # 3 PM New York
next_hour = pd.Timestamp(2024, 7, 21, 16, 0, 0, tz='America/New_York')  # 4 PM New York

for i in range(start_batch, num_batches):
    start_index = i * batch_size
    end_index = min((i + 1) * batch_size, num_rows)
    batch_data = []

    for j in tqdm(range(start_index, end_index), desc=f"Processing Batch {i + 1}/{num_batches}"):
        latitude = unique_locations_list[j]['Latitude']
        longitude = unique_locations_list[j]['Longitude']

        temps_3pm = get_hourly_temperatures(latitude, longitude, hour, next_hour)
        temps_4pm = get_hourly_temperatures(latitude, longitude, next_hour, next_hour)

        if temps_3pm and temps_4pm:
            temp_3pm = temps_3pm.get(hour)
            temp_4pm = temps_4pm.get(next_hour)

            batch_data.append({
                'Latitude': latitude,
                'Longitude': longitude,
                'Temperature_3PM': temp_3pm,
                'Temperature_4PM': temp_4pm
            })

    batch_df = pd.DataFrame(batch_data)
    file_name = f"temperature_batch_{i + 1}.csv"
    batch_df.to_csv(file_name, index=False)


11229


Processing Batch 4/88:  27%|██▋       | 34/128 [00:27<01:18,  1.19it/s]

Error fetching temperature for 40.80579, -73.92488167: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)


Processing Batch 5/88:  68%|██████▊   | 87/128 [01:11<00:33,  1.23it/s]

Error fetching temperature for 40.81611333, -73.930395: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)


Processing Batch 6/88:  90%|████████▉ | 115/128 [17:32<1:02:28, 288.31s/it]

Error fetching temperature for 40.82205, -73.92388833: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)


Processing Batch 8/88:  55%|█████▍    | 70/128 [00:57<00:47,  1.23it/s]

In [ ]:
# prompt: join all batches in one csv

import pandas as pd
import glob

# Assuming all your batch CSV files are in the same directory and follow a naming pattern.
# Replace 'batch_*.csv' with the actual pattern of your batch files.
all_files = glob.glob("batch_*.csv")

# Create an empty list to store dataframes
df_list = []

# Iterate over all the files and append them to the list
for filename in all_files:
    try:
        df = pd.read_csv(filename)
        df_list.append(df)
    except FileNotFoundError:
        print(f"Error: File not found at {filename}")
    except pd.errors.EmptyDataError:
        print(f"Warning: File '{filename}' is empty and will be skipped.")
    except Exception as e:
        print(f"An error occurred while processing file '{filename}': {e}")

# Concatenate all dataframes into a single dataframe
if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)

    # Save the combined dataframe to a new CSV file
    combined_df.to_csv("combined_data.csv", index=False)
    print("Combined data saved to combined_data.csv")
else:
    print("No valid batch files found to combine.")
